In [1]:
!pip install "xarray[complete]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 154.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.9/155.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.5/377.5 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2

In [4]:
import jax.numpy as jnp
import os
from heat3d import (
    run_heat3d
)

nbiter, diag_steps = 20, 40
device = 'tpu'
out_dir = 'data_python'
lx = 2.0 * jnp.pi
dt = 0.001
kappa = 1.0
nrepeats = 1

# Mapping of local directories to their desired names inside the zip
device_name = 'TPUv5' if device == 'tpu' else 'cpu'

for dtype in ["float32"]:
    for nx in [32, 64, 128, 256, 512]:
        for solver_type in [0, 1, 2]:
            # Run simulation
            run_heat3d(
                nx=nx,
                lx=lx,
                nbiter=nbiter,
                nrepeats=nrepeats,
                diag_steps=diag_steps,
                dt=dt,
                out_dir=out_dir,
                kappa=kappa,
                solver_type=solver_type,
                dtype=dtype,
            )

            # Rename the resulting file to the requested format
            old_filename = f'heat3d_{dtype}.txt'
            new_filename = f'heat3d_{device_name}_{dtype}_N{nx}_solver{solver_type}.txt'
            if os.path.exists(old_filename):
                os.rename(old_filename, new_filename)
                print(f'Renamed {old_filename} to {new_filename}')

Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N32_solver0.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N32_solver1.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N32_solver2.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N64_solver0.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N64_solver1.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N64_solver2.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N128_solver0.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N128_solver1.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N128_solver2.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N256_solver0.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N256_solver1.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N256_solver2.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N512_solver0.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N512_solver1.txt
Renamed heat3d_float32.txt to heat3d_TPUv5_float32_N51

In [5]:
import zipfile
import os
from google.colab import files

# Define the name of the output zip file
zip_filename = 'heat3d_results.zip'

# Mapping of local directories to their desired names inside the zip
dump_mapping = {
    'jaxpr_dump': f'heat3d_{device_name}_{dtype}_jaxpr_dump',
    'hlo_dump': f'heat3d_{device_name}_{dtype}_hlo_dump'
}

# Create a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Add .txt files from the current directory
    txt_files = [f for f in os.listdir('.') if f.startswith('heat3d_') and f.endswith('.txt')]
    for file in txt_files:
        zipf.write(file)

    # Add files from dump folders into renamed subdirectories
    for local_folder, zip_folder in dump_mapping.items():
        if os.path.exists(local_folder):
            for root, dirs, files_in_dir in os.walk(local_folder):
                for file in files_in_dir:
                    file_path = os.path.join(root, file)
                    # Construct the internal path: new_folder_name/filename
                    archive_path = os.path.join(zip_folder, file)
                    zipf.write(file_path, arcname=archive_path)

# Download the zip file
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>